# 03 · CV Pipeline — Segmentation Approach (v2)

Instance **segmentation** for insects (pixel mask + type) with **detection** for
flowers (separate box per flower). Counts each segmented insect entering a flower's
box into a per-flower / per-insect-type table with tracked IDs (no double-count on
fly-off + return).

Stages:
1. **SAM-bootstrapped seg dataset** — turn tight insect boxes into YOLO-seg masks.
2. **Insect segmenter** — fine-tune `yolo26s-seg` (single class `insect`).
3. **Flower detector** + **species classifier** — reused unchanged.
4. **Video** — flower box (ID) + insect mask (type, track ID) + debounced visit count.

Heavy logic lives in `src/cv_engine/`; this notebook orchestrates it and is
**idempotent** (existing outputs are reused, not rebuilt). Requires the CV env
(`bash scripts/setup_cv.sh`) and an NVIDIA GPU for training.

In [ ]:
import sys, os, json
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg"); import matplotlib.pyplot as plt

here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / "src" / "config.py").exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()
from src import config as C
from src.cv_engine import prepare_insect, sam_to_seg, train as cvtrain, video_seg
import torch
RUNS = C.INTERIM_DIR / "cv_runs"
print("device:", "cuda" if torch.cuda.is_available() else "cpu",
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
def exists(p): return Path(p).exists()

## 1 · SAM-bootstrapped segmentation dataset

Insect **mask** data is scarce, so we synthesize it: each already tight-boxed insect
image (one centered organism) is passed to **SAM** (MobileSAM) with a center-point
prompt; the mask is converted to a YOLO-seg polygon label. Single class `insect`
(the *type* comes from the species classifier downstream). Idempotent — skipped if
`data/interim/insect_seg/data.yaml` already exists.

In [ ]:
seg_yaml = C.INTERIM_DIR / "insect_seg" / "data.yaml"
if not seg_yaml.exists():
    # requires the boxed insect set (data/interim/insect_det1); build it if missing
    if not (C.INTERIM_DIR / "insect_det1" / "data.yaml").exists():
        prepare_insect.run(); prepare_insect.export_single_class()
    print(json.dumps(sam_to_seg.run(), indent=2))
else:
    print("seg dataset present:", seg_yaml)
n = {s: len(list((C.INTERIM_DIR / 'insect_seg' / 'labels' / s).glob('*.txt'))) for s in ('train','val','test')}
print("seg masks:", n)

## 2 · Insect segmenter — `yolo26s-seg`

Fine-tune YOLO26-seg on the SAM masks. Idempotent: loads existing `best.pt` if present.

In [ ]:
seg_run = RUNS / "insect_seg_yolo26s"
seg_w = seg_run / "weights" / "best.pt"
if not seg_w.exists():
    cvtrain.train(str(seg_yaml), "insect_seg_yolo26s", model="yolo26s-seg.pt",
                  epochs=80, imgsz=640, batch=8, patience=15)
print("segmenter weights:", seg_w, "|", exists(seg_w))

## 3 · Reused models

Flower detector (`yolo26n`, box mAP@0.5 0.917) and the iNaturalist species
classifier (top-1 0.962) are unchanged from the detection pipeline.

In [ ]:
flower_w = RUNS / "flower_yolo26n" / "weights" / "best.pt"
clf_w    = RUNS / "species_classifier" / "best.pt"
for label, w in [("flower", flower_w), ("classifier", clf_w)]:
    print(f"{label:10}", w, "|", exists(w))

## 4 · Metrics

In [ ]:
def best_seg_map(csv_path):
    df = pd.read_csv(csv_path); df.columns = [c.strip() for c in df.columns]
    i = df["metrics/mAP50-95(M)"].idxmax()
    return {k: round(float(df.loc[i, f"metrics/{k}"]), 4)
            for k in ["mAP50(M)", "mAP50-95(M)", "precision(M)", "recall(M)", "mAP50(B)"]}
metrics = {
    "insect_segmenter": best_seg_map(seg_run / "results.csv"),
    "flower_detector_mAP50(B)": round(pd.read_csv(RUNS / "flower_yolo26n" / "results.csv").rename(columns=str.strip)["metrics/mAP50(B)"].max(), 4),
}
print(json.dumps(metrics, indent=2))

## 5 · Video — segment insects, count flower visits

The incoming stream is resampled to **24 fps** (single real-time camera). For every
clip: flowers are detected per frame (separate box + stable ID), insects are
**instance-segmented + tracked** (BoT-SORT) with a **per-instance colour** (bee #1 ≠
bee #2), typed by the classifier, and a visit is counted when an insect **mask
overlaps a flower box** — each (insect, flower) pair **once ever**, so a fly-off +
return is not a second visit. Each visit is stamped with a **timestamp**. Annotated
videos (masks) + per-flower CSV + a per-visit timeline CSV are written to the root
`test_video_result/` folder.

In [ ]:
out = REPO / "test_video_result"; out.mkdir(exist_ok=True)
results = []
for v in sorted(C.TEST_VIDEO_DIR.glob("*.mp4")):
    r = video_seg.count_visits_seg(str(v), str(flower_w), str(seg_w), str(clf_w),
                                   out, save_video=True, flower_interval=5)
    results.append(r); print(v.name, "->", r["flowers"], "flowers,", sum(r["visits"].values()), "visits @", r["out_fps"], "fps")
print("saved annotated (mask) videos to", out)

### Visit tallies (per flower, per insect type)

In [ ]:
frames = []
for csvf in sorted(out.glob("*_visits.csv")):
    d = pd.read_csv(csvf); d.insert(0, "video", csvf.stem.replace("_visits", ""))
    frames.append(d)
visits_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
visits_df[visits_df["total"] > 0] if not visits_df.empty else visits_df

### Visit timeline (per-visit timestamps, seconds into clip)

In [ ]:
tls = []
for tlf in sorted(out.glob("*_timeline.csv")):
    d = pd.read_csv(tlf)
    if len(d):
        d.insert(0, "video", tlf.stem.replace("_timeline", "")); tls.append(d)
timeline_df = pd.concat(tls, ignore_index=True) if tls else pd.DataFrame()
timeline_df.head(20)

### Sample annotated frame (flower boxes + insect masks)

In [ ]:
import cv2, glob
vids = sorted(glob.glob(str(out / "*_annotated.mp4")))
if vids:
    cap = cv2.VideoCapture(vids[0])
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, fr = cap.read(); cap.release()
    if ok:
        plt.figure(figsize=(10, 6)); plt.axis("off")
        plt.title(Path(vids[0]).stem)
        plt.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); plt.show()

## 6 · Summary

In [ ]:
print(json.dumps({"metrics": metrics,
                  "videos": len(results),
                  "total_visits": int(sum(sum(r["visits"].values()) for r in results))},
                 indent=2))